In [1]:
import os
import re
from pathlib import Path

def thai_to_arabic(text):
    if not text: return text
    thai_digits = "๐๑๒๓๔๕๖๗๘๙"
    arabic_digits = "0123456789"
    trans = str.maketrans(thai_digits, arabic_digits)
    return text.translate(trans)

def extract_file_info(filename):
    sub_district_match = re.search(r'(?:ตำบล|ต\.)\s*([\u0E00-\u0E7F]+)', filename)
    if sub_district_match:
        sub_district = sub_district_match.group(1)
    else:
        set_match = re.search(r'(ชุดที่\s*\d+)', filename)
        sub_district = set_match.group(1) if set_match else "-1"
        
    unit_match = re.search(r'หน่วยที่_?(\d+)', filename)
    unit_number = unit_match.group(1) if unit_match else "-1"

    return sub_district, unit_number

# ใช้เวอร์ชันอ่าน HTML Table (ลบเวอร์ชันเก่าทิ้ง)
def extract_with_regex(content):
    # 1. แปลงเลขไทยเป็นอารบิกทั้งหมดก่อน
    content = thai_to_arabic(content)
    
    # ลบลูกน้ำในตัวเลข (เช่น 1,200 -> 1200) เพื่อให้ดึงง่ายขึ้น
    content = re.sub(r'(\d),(\d)', r'\1\2', content)

    # 2. ดึงจำนวนผู้มีสิทธิ และ ผู้มาแสดงตน (อิงตามตัวอย่างเอกสาร)
    eligible_match = re.search(r'บัญชีรายชื่อผู้มีสิทธิเลือกตั้ง.*?จำนวน\s*(\d+)', content)
    voted_match = re.search(r'ที่มาแสดงตน.*?จำนวน\s*(\d+)', content)

    eligible_voters = eligible_match.group(1) if eligible_match else "-1"
    actual_voters = voted_match.group(1) if voted_match else "-1"

    # 3. ดึงชื่อและคะแนนจาก HTML Table
    scores = []
    
    rows = re.findall(r'<tr.*?>(.*?)</tr>', content, re.IGNORECASE | re.DOTALL)
    
    for row in rows:
        cells = re.findall(r'<td.*?>(.*?)</td>', row, re.IGNORECASE | re.DOTALL)
        # ลบ HTML Tags อื่นๆ ที่อาจแทรกอยู่ในเซลล์ และตัดช่องว่าง
        cells = [re.sub(r'<.*?>', '', c).strip() for c in cells]
        
        # ถ้าคอลัมน์แรกเป็น "ตัวเลข" (แสดงว่าเป็นแถวรายชื่อ)
        if cells and re.match(r'^\d+$', cells[0]):
            if len(cells) == 4: # บช
                name = cells[1]
                score = cells[2]
                scores.append((name, score))
            elif len(cells) >= 5: # เขต
                name = cells[1] 
                score = cells[3]
                scores.append((name, score))

    return eligible_voters, actual_voters, scores

def main():
    source_dir = Path("ocr-result")
    output_dir = Path("ocr-result1")
    output_dir.mkdir(exist_ok=True)

    csv1_path = output_dir / "master_summary.csv"
    csv2_path = output_dir / "master_results.csv"

    header1 = "type,province,district,sub-district,unit_number,จำนวนผู้มีสิทธิมาเลือกตั้ง,จำนวนผู้มาแสดงตน\n"
    header2 = "type,province,district,sub-district,unit_number,name,score\n"

    if not csv1_path.exists():
        with open(csv1_path, 'w', encoding='utf-8') as f:
            f.write(header1)
    if not csv2_path.exists():
        with open(csv2_path, 'w', encoding='utf-8') as f:
            f.write(header2)

    files = sorted(list(source_dir.glob("**/*.txt")))
    print(f"พบไฟล์ทั้งหมด: {len(files)} ไฟล์")

    for file in files:
        try:
            with open(file, 'r', encoding="utf-8") as f:
                content = f.read()

            structure = file.parent.parts
            district = structure[1] if len(structure) > 1 else "ไม่ระบุ"
            province = "ลำปาง"
            
            sub_district, unit_num = extract_file_info(file.stem)
            if sub_district == "-1" and len(structure) > 2:
                sub_district = structure[2]

            type_val = "บช" if "บช" in file.name else "เขต"

            eligible, voted, scores = extract_with_regex(content)

            row1 = f"{type_val},{province},{district},{sub_district},{unit_num},{eligible},{voted}\n"
            with open(csv1_path, "a", encoding="utf-8") as fw1:
                fw1.write(row1)

            with open(csv2_path, "a", encoding="utf-8") as fw2:
                if not scores:
                    fw2.write(f"{type_val},{province},{district},{sub_district},{unit_num},-1,-1\n")
                else:
                    for name, score in scores:
                        row2 = f"{type_val},{province},{district},{sub_district},{unit_num},{name},{score}\n"
                        fw2.write(row2)

            print(f"✅ สำเร็จ: {file.stem} (พบ {len(scores)} รายชื่อ)")

        except Exception as e:
            print(f"❌ Error on {file.stem}: {e}")


In [2]:
main()

พบไฟล์ทั้งหมด: 705 ไฟล์
✅ สำเร็จ: ชุดที่ 1 ส.ส. 5-17(บช) (พบ 57 รายชื่อ)
✅ สำเร็จ: ชุดที่ 1 ส.ส. 5-17 (พบ 11 รายชื่อ)
✅ สำเร็จ: ชุดที่ 10 5-17(บช) (พบ 57 รายชื่อ)
✅ สำเร็จ: ชุดที่ 10 5-17 (พบ 11 รายชื่อ)
✅ สำเร็จ: ชุดที่ 11 5-17(บช) (พบ 57 รายชื่อ)
✅ สำเร็จ: ชุดที่ 11 5-17 (พบ 11 รายชื่อ)
✅ สำเร็จ: ชุดที่ 12 5-17 (บช) (พบ 57 รายชื่อ)
✅ สำเร็จ: ชุดที่ 12 5-17 (พบ 11 รายชื่อ)
✅ สำเร็จ: ชุดที่ 13 5-17 (บช) (พบ 57 รายชื่อ)
✅ สำเร็จ: ชุดที่ 13 5-17 (พบ 11 รายชื่อ)
✅ สำเร็จ: ชุดที่ 2 ส.ส. 5-17 (บช) (พบ 57 รายชื่อ)
✅ สำเร็จ: ชุดที่ 2 ส.ส. 5-17 (พบ 11 รายชื่อ)
✅ สำเร็จ: ชุดที่ 3 ส.ส. 5-17 (บช) (พบ 57 รายชื่อ)
✅ สำเร็จ: ชุดที่ 3 ส.ส. 5-17 (พบ 11 รายชื่อ)
✅ สำเร็จ: ชุดที่ 4 ส.ส. 5-17(บช) (พบ 57 รายชื่อ)
✅ สำเร็จ: ชุดที่ 4 ส.ส. 5-17 (พบ 11 รายชื่อ)
✅ สำเร็จ: ชุดที่ 5 ส.ส. 5-17(บช) (พบ 57 รายชื่อ)
✅ สำเร็จ: ชุดที่ 5 ส.ส. 5-17 (พบ 11 รายชื่อ)
✅ สำเร็จ: ชุด 6 ส.ส. 5-17 บช (พบ 57 รายชื่อ)
✅ สำเร็จ: ชุด 6 ส.ส. 5-17 (พบ 11 รายชื่อ)
✅ สำเร็จ: ชุดที่ 7 ส.ส. 5-17 (บช) (พบ 57 รายชื่อ)
✅ สำเร็จ: ชุดที่ 7 ส.